# tech-history 음성 생산 노트북 (Chatterbox Multilingual, 무료 T4 GPU)

**사용법**
1. 상단 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 저장
2. 아래 `EPISODE` 값 확인 (기본 "01")
3. 런타임 → **모두 실행** (첫 실행은 모델 다운로드로 5~10분 소요)
4. 마지막 셀이 끝나면 `voice_01.zip` 이 자동 다운로드됨
5. 압축을 `tech-history/video/output/01_v2/audio/` 폴더에 풀기

(선택) 본인 목소리로 낭독시키려면: 왼쪽 폴더 아이콘 → `ref.wav`(5~10초 육성 녹음) 업로드 후 모두 실행 — 자동으로 그 목소리를 복제해 낭독합니다.

In [ ]:
EPISODE = "01"  # 생산할 편 번호
!pip install -q chatterbox-tts requests

In [ ]:
import json, os, requests, torch, torchaudio
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

url = f"https://raw.githubusercontent.com/nous-zero/tech-history/main/video/scripts/{EPISODE}.json"
script = requests.get(url).json()
print("대본:", script["title"], "/ 문단", len(script["segments"]))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("장치:", device)
model = ChatterboxMultilingualTTS.from_pretrained(device=device)

In [ ]:
REF = "ref.wav" if os.path.exists("ref.wav") else None  # 육성 복제용(선택)
out_dir = f"voice_{EPISODE}"
os.makedirs(out_dir, exist_ok=True)
for seg in script["segments"]:
    kwargs = {"language_id": "ko"}
    if REF:
        kwargs["audio_prompt_path"] = REF
    wav = model.generate(seg["text"], **kwargs)
    path = os.path.join(out_dir, f"seg{seg['id']:03d}.wav")
    torchaudio.save(path, wav, model.sr)
    print(f"seg{seg['id']:03d} 완료 ({wav.shape[-1]/model.sr:.1f}초)")
print("전체 합성 완료")

In [ ]:
import shutil
zip_path = shutil.make_archive(f"voice_{EPISODE}", "zip", out_dir)
from google.colab import files
files.download(zip_path)